# PGNet — tự kiểm tra trên Colab T4, từng bước

**Chưa chạy:** các bảng, ảnh và trạng thái bên dưới chỉ xuất hiện khi bạn chạy cell bằng nút ▶.
Mục tiêu: xác minh weights/runtime thực sự forward trên **9 ảnh train**. Không có ảnh test hoặc GT trong gói; không chấm Hmean/CER và không chọn winner.

1. Colab → **File → Upload notebook** → chọn notebook này.
2. **Runtime → Change runtime type → T4 GPU → Save**, rồi Connect.
3. Chạy lần lượt các cell từ trên xuống. Upload ZIP qualification nguyên bản khi được hỏi; không giải nén trước.
4. Xem ảnh, log, bảng receipt và prediction. Chỉ chuyển sang comparison khi bảng gate đều PASS.
5. **File → Save**, lưu/download notebook có outputs, rồi tải ZIP evidence ở cell cuối. Nếu lỗi, vẫn chạy riêng cell export.

Không đổi weights/config/threshold/dictionary/grouping. Receipt là bằng chứng chính; ảnh overlay chỉ giúp quan sát output native.

Runner đã sửa bootstrap và giữ đường dẫn driver NVIDIA; không cần copy cell vá bên ngoài. Upload đúng ZIP canonical bên dưới. Nếu dùng lại runtime chưa có PASS, notebook giữ log lỗi và kiểm lại cache đã cài.


In [ ]:
from pathlib import Path
import sys, subprocess, json, hashlib, zipfile
from google.colab import files
hardware = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True).strip()
print(hardware)
assert 'T4' in hardware, 'Chọn T4 và khởi động lại runtime trước khi tiếp tục.'
ROOT = Path('/content/joint-qualification/runner')
WORK = Path('/content/joint-qualification/work')
OUT = Path('/content/joint-qualification/evidence')
OUT.mkdir(parents=True, exist_ok=True)
(OUT/'hardware.json').write_text(json.dumps({'gpu':hardware,'kernel_python':sys.version,'model_execution_verified':False}, indent=2))

from IPython.display import display, HTML
from html import escape
def show_table(rows, columns, title):
    markup = '<h3>' + escape(title) + '</h3><div style="overflow:auto"><table border="1" cellpadding="6"><thead><tr>'
    markup += ''.join('<th>' + escape(str(k)) + '</th>' for k in columns) + '</tr></thead><tbody>'
    markup += ''.join('<tr>' + ''.join('<td style="white-space:pre-wrap;word-break:break-word">' + escape(str(r.get(k, ''))) + '</td>' for k in columns) + '</tr>' for r in rows)
    markup += '</tbody></table></div>'
    display(HTML(markup))
    return markup
show_table([{'GPU': hardware, 'Python kernel': sys.version.split()[0], 'Trạng thái': 'HARDWARE ONLY — model chưa chạy'}], ['GPU', 'Python kernel', 'Trạng thái'], 'Bước 1 — Phần cứng')


## Upload gói đã khóa

Gói khoảng 3,9 MB gồm code và đúng 9 ảnh train. Bạn tự chọn file upload. Cell kiểm SHA256 và mọi member trước khi giải nén.


In [ ]:
uploaded = files.upload()
assert len(uploaded) == 1, 'Chọn đúng một ZIP qualification.'
filename, raw = next(iter(uploaded.items()))
assert hashlib.sha256(raw).hexdigest() == 'ee6c45d306f9684299cc0eb0492ffe962b1f3ac03cc0b31c2d3019456e01eb0a', 'Gói ZIP khác bản đã khóa.'
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(filename) as archive:
    manifest = json.loads(archive.read('qualification-manifest.json'))
    assert manifest['test_inputs'] == 0 and len(manifest['train_inputs']) == 9
    names = archive.namelist()
    assert len(names) == len(set(names))
    assert set(names) == set(manifest['files']) | {'qualification-manifest.json'}
    for name, expected in manifest['files'].items():
        relative = Path(name)
        assert not relative.is_absolute() and '..' not in relative.parts and chr(92) not in name and ':' not in name
        assert hashlib.sha256(archive.read(name)).hexdigest() == expected
    # Preserve prior failure evidence before Step4 writes a fresh log.
    from datetime import datetime, timezone
    import shutil
    prior_manifest_path = ROOT/'qualification-manifest.json'
    if prior_manifest_path.is_file():
        prior_manifest = json.loads(prior_manifest_path.read_text())
        assert prior_manifest['train_inputs'] == manifest['train_inputs'], 'Train cohort changed.'
        if (OUT/'pgnet_totaltext_joint_t4/preflight.json').exists():
            assert prior_manifest == manifest, 'Passed run cannot be changed; choose a new output root.'
        elif (OUT/'qualification.log').is_file():
            history = OUT/'failure-history'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
            history.mkdir(parents=True, exist_ok=False)
            for previous in ['qualification.log','process-status.json','cuda-diagnostics.json','qualification-manifest.json']:
                if (OUT/previous).is_file():
                    shutil.copy2(OUT/previous, history/previous)
    archive.extractall(ROOT)
print('Verified:', len(manifest['train_inputs']), 'train frames; zero test frames')

show_table([{'File':filename, 'Bytes':len(raw), 'SHA256':hashlib.sha256(raw).hexdigest(), 'Train':len(manifest['train_inputs']), 'Test':manifest['test_inputs'], 'Kết quả':'PASS — đúng gói đã khóa'}], ['File','Bytes','SHA256','Train','Test','Kết quả'], 'Bước 2 — Kiểm tra gói')
(OUT/'qualification-manifest.json').write_text(json.dumps(manifest, indent=2))


In [ ]:
from PIL import Image, ImageDraw
VIS = OUT/'visualizations'
VIS.mkdir(parents=True, exist_ok=True)
preview_rows = []
sheet = Image.new('RGB', (1200, 3*270), 'white')
for index, row in enumerate(manifest['train_inputs']):
    with Image.open(ROOT/row['frame_id']) as image:
        image = image.convert('RGB')
        preview_rows.append({'Frame ID':row['frame_id'], 'Width':image.width, 'Height':image.height, 'SHA256':row['image_sha256']})
        image.thumbnail((390, 235))
        x, y = (index % 3)*400, (index // 3)*270
        sheet.paste(image, (x, y+30))
        ImageDraw.Draw(sheet).text((x+6,y+6), f'Train frame {index+1}', fill='black')
sheet.save(VIS/'train-inputs.jpg')
display(sheet)
show_table(preview_rows, ['Frame ID','Width','Height','SHA256'], 'Bước 3 — Đủ 9 ảnh train, không có test')


## Bạn tự chạy cài môi trường và forward pass

Tạo môi trường Python 3.10.16 riêng, tải checkpoint PGNet chính thức đã khóa SHA, cài Paddle GPU/CUDA 11.8/cuDNN 8 riêng và chạy 9 ảnh train. Không hạ cấp Python hoặc CUDA của kernel Colab. Log được giữ cả khi thất bại.


In [ ]:
import time, os, signal
command = [sys.executable, '-m', 'experiments.scripts.qualify_joint_spotting', '--root', str(ROOT), '--work', str(WORK), '--output', str(OUT)]
log = OUT/'qualification.log'
qualification_returncode = None
print('Bước 4 — Đang cài môi trường riêng và forward; cell có thể chạy lâu. Không bấm chạy lại khi còn hoạt động.')
print('Log đầy đủ:', log)
handle = display(HTML('<b>STARTING — chưa có kết quả model</b>'), display_id=True)
tail, last_update = [], 0.0
with log.open('w') as stream:
    process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            stream.write(line)
            stream.flush()
            tail.append(line)
            tail = tail[-40:]
            if time.monotonic()-last_update > 1:
                handle.update(HTML('<b>RUNNING — 40 dòng cuối; toàn bộ log được giữ</b><pre>' + escape(''.join(tail)) + '</pre>'))
                last_update = time.monotonic()
        qualification_returncode = process.wait()
    except KeyboardInterrupt:
        # Stop the process tree on Colab Linux; avoid orphan dependency/model jobs.
        os.killpg(process.pid, signal.SIGTERM)
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        raise
handle.update(HTML('<b>' + ('PROCESS COMPLETED' if qualification_returncode == 0 else 'FAILED — giữ evidence, chưa chạy test') + '</b><pre>' + escape(''.join(tail)) + '</pre>'))
(OUT/'process-status.json').write_text(json.dumps({'returncode':qualification_returncode, 'model_pass_requires_receipt':True}, indent=2))
assert qualification_returncode == 0, 'Preflight chưa đạt. Chạy cell export cuối để giữ log; chưa chạy test.'


## Xác nhận receipt

Chỉ coi GPU train preflight đạt khi receipt ghi `passed`, device T4 và đủ 9 train observations. Prediction geometry lỗi được ghi lại, không lọc; protocol test sẽ tính chúng là FP. Kết quả này chưa phải accuracy hoặc benchmark test.


In [ ]:
receipt_path = OUT/'pgnet_totaltext_joint_t4/preflight.json'
receipt = json.loads(receipt_path.read_text()) if receipt_path.is_file() else {}
observations = receipt.get('train_observations', [])
actual_ids = [r['frame_id'] for r in observations]
expected_ids = [r['frame_id'] for r in manifest['train_inputs']]
expected_hashes = {r['frame_id']:r['image_sha256'] for r in manifest['train_inputs']}
checks = [
    ('Process vừa chạy thành công', globals().get('qualification_returncode') == 0),
    ('Receipt tồn tại', receipt_path.is_file()),
    ('status = passed', receipt.get('status') == 'passed'),
    ('Thiết bị model là T4', 'T4' in receipt.get('runtime', {}).get('device','')),
    ('Đủ đúng 9 frame ID đã khóa, không trùng', len(actual_ids) == 9 and len(set(actual_ids)) == 9 and set(actual_ids) == set(expected_ids)),
    ('Hash ảnh đúng manifest', bool(observations) and all(r.get('image_sha256') == expected_hashes.get(r['frame_id']) for r in observations)),
    ('Chỉ frame train', bool(actual_ids) and all(x.startswith('Frames/train/') for x in actual_ids)),
    ('no_test_inference = true', receipt.get('no_test_inference') is True),
    ('accuracy_not_evaluated = true', receipt.get('accuracy_not_evaluated') is True),
]
gate_rows = [{'Kiểm tra':name, 'Kết quả':'PASS' if passed else 'FAIL'} for name, passed in checks]
gate_html = show_table(gate_rows, ['Kiểm tra','Kết quả'], 'Bước 5 — Receipt gate')
(OUT/'receipt-gates.html').write_text(gate_html, encoding='utf-8')
(OUT/'receipt-gates.json').write_text(json.dumps(gate_rows, ensure_ascii=False, indent=2), encoding='utf-8')
display(HTML('<details><summary>Runtime và binding đầy đủ</summary><pre>' + escape(json.dumps({'runtime':receipt.get('runtime'), 'binding':receipt.get('binding')}, ensure_ascii=False, indent=2)) + '</pre></details>'))
assert all(passed for _, passed in checks), 'Gate FAIL: giữ evidence; chưa chuyển sang full test.'
print('PASS: actual T4 train-forward đủ 9 ảnh. Đây chưa phải kết quả accuracy hoặc benchmark test.')


In [ ]:
import csv, math
summary, all_predictions, html_tables = [], [], []
for index, row in enumerate(observations):
    predictions = row['predictions']
    summary.append({'Frame':index+1, 'Frame ID':row['frame_id'], 'Regions':len(predictions), 'Invalid native':row['invalid_predictions'], 'Wall seconds':round(row['wall_seconds'],3)})
    with Image.open(ROOT/row['frame_id']) as image:
        overlay = image.convert('RGB')
    draw = ImageDraw.Draw(overlay)
    frame_rows = []
    for region_id, prediction in enumerate(predictions, 1):
        points = prediction['polygon']
        renderable = len(points) >= 3 and all(len(p) == 2 and all(math.isfinite(v) for v in p) for p in points)
        if renderable:
            xy = [tuple(p) for p in points]
            draw.line(xy+[xy[0]], fill='#008bff', width=3)
            draw.text(xy[0], str(region_id), fill='#ff2020', stroke_width=1, stroke_fill='white')
        item = {'Frame ID':row['frame_id'], 'Region':region_id, 'Text':prediction['text'], 'Polygon':json.dumps(points), 'Overlay':'drawn' if renderable else 'unrenderable — retained'}
        frame_rows.append(item)
        all_predictions.append(item)
    overlay.save(VIS/f'frame-{index+1:02d}-predictions.png')
    display(HTML('<h3>Frame ' + str(index+1) + ': ' + escape(row['frame_id']) + '</h3>'))
    display(HTML('<p>Native invalid polygons: ' + str(row['invalid_predictions']) + '. Overlay không sửa/lọc prediction; số đỏ tra bảng bên dưới. Không có GT hoặc confidence giả.</p>'))
    display(overlay.resize((min(1200,overlay.width), max(1,round(overlay.height*min(1200,overlay.width)/overlay.width)))))
    html_tables.append(show_table(frame_rows, ['Region','Text','Polygon','Overlay'], 'Toàn bộ transcription native — ' + str(len(predictions)) + ' vùng'))
summary_html = show_table(summary, ['Frame','Frame ID','Regions','Invalid native','Wall seconds'], 'Bước 6 — Tổng hợp 9 frame')
display(HTML('<p>Wall seconds gồm chi phí của từng lời gọi trong preflight; không dùng làm benchmark tốc độ ổn định.</p>'))
# Dependency-free count chart. These counts measure output coverage, not accuracy.
chart = Image.new('RGB', (900,380), 'white')
draw = ImageDraw.Draw(chart)
draw.text((15,10), 'Native region counts per train frame (not accuracy)', fill='black')
maximum = max([r['Regions'] for r in summary]+[1])
for i, row in enumerate(summary):
    x, top = 35+i*95, 320-round(260*row['Regions']/maximum)
    draw.rectangle((x,top,x+50,320), fill='#1678b8')
    draw.text((x,top-18), str(row['Regions']), fill='black')
    draw.text((x,335), str(row['Frame']), fill='black')
chart.save(VIS/'region-counts.png')
display(chart)
(VIS/'prediction-tables.html').write_text('<meta charset="utf-8">'+summary_html+''.join(html_tables), encoding='utf-8')
with (VIS/'predictions.csv').open('w', newline='', encoding='utf-8-sig') as stream:
    writer = csv.DictWriter(stream, fieldnames=['Frame ID','Region','Text','Polygon','Overlay'])
    writer.writeheader()
    writer.writerows(all_predictions)
(VIS/'frame-summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã lưu 9 overlays, chart, toàn bộ text/polygon CSV/HTML. Không chấm accuracy; giữ nguyên prediction native lỗi.')


## Lưu evidence kể cả khi preflight thất bại

Có thể chạy riêng cell này để tải log và partial receipts. Sau khi xác minh đạt, chuyển sang notebook comparison v2; notebook đó vẫn phải tạo preflight binding riêng cho run test.


In [ ]:
import shutil
# Safe to run separately after a failure, provided the hardware cell created OUT.
evidence_files = []
for item in sorted(OUT.rglob('*')):
    if item.is_file() and item.name != 'evidence-manifest.json':
        evidence_files.append({'path':item.relative_to(OUT).as_posix(), 'bytes':item.stat().st_size, 'sha256':hashlib.sha256(item.read_bytes()).hexdigest()})
(OUT/'evidence-manifest.json').write_text(json.dumps({'files':evidence_files, 'passed_receipt_present':(OUT/'pgnet_totaltext_joint_t4/preflight.json').is_file(), 'note':'Partial evidence is not a passed preflight; inspect receipt-gates.json.'}, indent=2))
archive_path = shutil.make_archive('/content/pgnet-t4-preflight-evidence', 'zip', OUT)
show_table(evidence_files, ['path','bytes','sha256'], 'Bước 7 — Evidence sẽ tải (kể cả khi lỗi)')
print('ZIP:', archive_path, 'Bytes:', Path(archive_path).stat().st_size)
print('Nhớ File → Save / Download .ipynb để giữ các outputs hiển thị trong notebook.')
files.download(archive_path)
